# 第 16 课 · 显存、吞吐与训练瓶颈

“GPU 可用”与“GPU 被有效利用”是两件事。本课练习建立资源模型、正确计时，并识别微基准的适用范围。需要 CUDA；无需前课权重。8GB 卡从小尺寸开始，先关闭其他训练内核。

## 1 · 先做显存预算，再启动训练

普通混合精度 AdamW 的一种常见估算是：BF16 参数 2P 字节、梯度 2P、FP32 master 参数 4P、两份 FP32 优化器状态 8P，合计约 16P。具体框架可能使用 FP32 梯度、不同 optimizer state dtype 或没有独立 master copy，必须检查实际实现。

这还不含激活、临时工作区、通信缓冲、碎片、rollout KV cache 和额外模型副本。KV cache 的简化估算为

$$2\times L\times N_{seq}\times T\times H_{kv}\times d_{head}\times bytes.$$

其中 2 表示 K 与 V。GQA 使用 KV head 数，不是 query head 数。vLLM 的分页、块对齐与并行切分会进一步影响实际占用。


In [ ]:
import gc, json, statistics, time
from pathlib import Path
import pandas as pd
import torch
from IPython.display import display
assert torch.cuda.is_available(), '本课需要 CUDA'
GiB = 1024**3
def budget(parameters, layers, sequences, tokens, kv_heads, head_dim, bytes_per_element=2):
    return dict(parameters=parameters, adamw_example_gib=16*parameters/GiB,
        kv_cache_example_gib=2*layers*sequences*tokens*kv_heads*head_dim*bytes_per_element/GiB)
display(pd.DataFrame([budget(p,24,8,2048,8,64) for p in [1250000,500000000,7000000000]]))
print('上表是模型示例预算，未包含激活与进程开销，不是可运行配置保证。')
print(torch.cuda.get_device_name(0))


## 2 · CUDA 计时与批量大小

CPU 提交 CUDA 操作后可能立即返回。用 `torch.cuda.Event` 记录设备时间，并在读取前同步；另记同步后的墙钟时间。先预热，再重复测量，报告中位数。

本格是固定权重 MLP 的推理微基准，用 batch=1/32/128 比较样本吞吐与批次延迟。它不包含 tokenizer、数据加载、网络和自回归解码，不能用来宣称真实服务 token/s。关闭 TF32 以明确 FP32 计算设置，结束时恢复原设置。


In [ ]:
torch.manual_seed(16016)
original_tf32 = torch.backends.cuda.matmul.allow_tf32
torch.backends.cuda.matmul.allow_tf32 = False
net = torch.nn.Sequential(torch.nn.Linear(512,2048),torch.nn.GELU(),torch.nn.Linear(2048,512)).cuda().eval()
results=[]
try:
    with torch.inference_mode():
        for batch_size in [1,32,128]:
            x=torch.randn(batch_size,512,device='cuda')
            for _ in range(10): net(x)
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
            samples=[]; walls=[]
            for _ in range(7):
                start,end=torch.cuda.Event(enable_timing=True),torch.cuda.Event(enable_timing=True)
                t=time.perf_counter(); start.record()
                for _ in range(50): net(x)
                end.record(); end.synchronize()
                walls.append((time.perf_counter()-t)*1000/50)
                samples.append(start.elapsed_time(end)/50)
            latency=statistics.median(samples)
            results.append(dict(batch=batch_size,device_batch_ms=latency,
                wall_batch_ms=statistics.median(walls),samples_per_second=batch_size/(latency/1000),
                process_peak_allocated_mib=torch.cuda.max_memory_allocated()/1024**2,
                process_reserved_mib=torch.cuda.memory_reserved()/1024**2))
finally:
    torch.backends.cuda.matmul.allow_tf32=original_tf32
display(pd.DataFrame(results))


## 3 · 混合精度的收益要与误差一起测

BF16 的指数范围接近 FP32，但尾数精度较低。FP16 的范围更窄，训练时常需要 loss scaling；具体设置取决于模型与框架。AMP autocast 只选择运算 dtype，不自动把所有模型状态都变小。

本格只比较一次 BF16 autocast 推理的数值误差，不据此断言端到端训练精度或速度。训练验证还应检查 loss、梯度范数、NaN/Inf 与固定验证集。


In [ ]:
if torch.cuda.is_bf16_supported():
    with torch.inference_mode():
        fp32=net(x)
        with torch.autocast('cuda',dtype=torch.bfloat16): mixed=net(x)
    relative_l2=float((mixed.float()-fp32).norm()/fp32.norm().clamp_min(1e-12))
    assert torch.isfinite(mixed).all()
    print('BF16 相对 L2 误差:', relative_l2)
else:
    relative_l2=None
    print('该 GPU 不支持 BF16，跳过本格比较。')
out=Path('experiment16_output');out.mkdir(exist_ok=True)
pd.DataFrame(results).to_csv(out/'microbenchmark.csv',index=False)
(out/'environment.json').write_text(json.dumps(dict(torch=torch.__version__,cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),bf16_relative_l2=relative_l2,
    scope='MLP inference microbenchmark; not serving latency or training throughput.'),indent=2))
del net,x
if relative_l2 is not None: del fp32,mixed
gc.collect();torch.cuda.empty_cache()


## 4 · 从日志定位问题，再选优化手段

| 现象 | 先确认的证据 | 可能的下一步 |
|---|---|---|
| CUDA OOM | 哪个进程、阶段与张量分配失败，allocated/reserved | 减少序列或 micro batch，再评估 checkpointing/offload |
| Ray worker 被杀 | 主机 RAM、系统日志、worker 数 | 降低并发 worker，减少重复加载 |
| GPU 利用率低 | 数据等待、CPU tokenizer、同步点与小 kernel | 用 profiler 区分 CPU/CUDA 时间，再改批量或流水线 |
| rollout 耗时占比高 | 输入/输出长度、每题采样数、prefill/decode 时间 | 调整采样预算或生成批次，同时复核评估协议 |
| 更新不稳定 | 梯度、裁剪比例、KL、奖励分布、有效组比例 | 先查 mask/log-prob/数据，再改学习率 |

进程的 `max_memory_allocated` 不是整张卡的总占用；`empty_cache()` 也不能释放仍被变量引用的张量。端到端测量要记录模型加载与 warm-up，并说明是否包含它们。

## 5 · LoRA、分布式与服务：下一阶段如何验收

LoRA 学习低秩增量 $\Delta W=BA$，减少可训练参数、梯度及优化器状态，但基础权重、激活和 KV cache 仍占资源。量化 LoRA 还引入量化误差与算子兼容问题。不要把“可训练参数少”直接换算成“整个作业显存同倍下降”。

数据并行复制模型处理不同样本；FSDP/ZeRO 切分参数或训练状态；张量并行切分层内计算；流水线并行切分层。它们交换的是不同张量，需要评估通信开销、负载均衡和失败恢复。单卡 NO_SHARD 不能证明多卡配置正确。

服务侧分别测首 token 延迟（TTFT）、后续 token 间隔（TPOT）、端到端延迟、并发下 p50/p95 与吞吐，并固定输入/输出长度和硬件。单请求快不意味着高并发稳定。

课后交付：为第 14 课画出进程与显存归属图，记录一次真实训练的时间分解。选择一个瓶颈，只改一个参数，比较质量、耗时和峰值内存。完整的 LoRA、多卡和服务实验尚未包含在本课程的实测范围内，下一阶段验收要求见 [工程学习路线](ENGINEERING_ROADMAP.md)。

参考：[PyTorch profiler](https://pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)、[AMP](https://pytorch.org/docs/stable/amp.html)、[LoRA](https://arxiv.org/abs/2106.09685)、[verl 性能调优](https://verl.readthedocs.io/en/latest/perf/perf_tuning.html)。
